# Kenzan: VLM on held-out v3 (Colab)

Runtime → Change runtime type → **T4 GPU**. Default: Qwen3-VL-8B-Instruct, 4-bit weights, **fp16 compute** (the T4 has no native bf16).

Held-out v3 is **8 new degradations of the same reference page**, not new documents.

This notebook only produces raw model responses (`vlm_cache/`); scoring is offline (`stress/run_vlm_compare.py`). Run the cells in order. **Step 3 is a smoke test on the clean reference page; if it reads fewer than 70 of 74 cells correctly (path-keyed), stop and download `vlm_cache.zip` (step 5) — do not run step 4.** The sha256 of every cache file and of the zip is printed here, at the source.

## 1. GPU, code (pinned), Tesseract, versions

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!git clone -q https://github.com/vedantjhunjhunwal/Kenzan.git
%cd Kenzan
# pinned: the frozen commit for this VLM run (pushed before any output of this run existed)
!git checkout -q __FROZEN_COMMIT__
!git rev-parse HEAD
!pip install -q -r requirements-vlm.txt
# second reading (A8/A9) needs Tesseract + the same tessdata_best jpn model as the scorer
!apt-get -qq install -y tesseract-ocr > /dev/null && bash scripts/setup_tesseract.sh > /dev/null
%env TESSDATA_PREFIX=/content/Kenzan/tessdata
!tesseract --version | head -1 && sha256sum tessdata/jpn.traineddata
import transformers, torch, huggingface_hub
print('transformers', transformers.__version__, '| torch', torch.__version__, '| huggingface_hub', huggingface_hub.__version__)
from huggingface_hub import model_info
print('hub revision of Qwen/Qwen3-VL-8B-Instruct:', model_info('Qwen/Qwen3-VL-8B-Instruct').sha)

## 2. Held-out pages
Regenerate the 8 pages and check their image hashes. If any line says MISMATCH, upload `heldout_v3_pdfs.zip` with the file browser and run the commented unzip line instead.

In [ ]:
!python stress/make_pdfs.py stress/specs/heldout_v3.json stress/out_heldout_v3/pdfs
# only if MISMATCH:
# !unzip -o ../heldout_v3_pdfs.zip -d stress/out_heldout_v3/pdfs

## 3. Smoke test (clean reference page at 300 dpi; not a held-out page)
Loads the model (~5 min download) and reads one page. Gate: at least 70 of 74 cells correct, path-keyed (value-correct cells are printed next to it). The printed `model_revision` must not be null.

In [ ]:
!python scripts/vlm_run.py --smoke-only --cache vlm_cache --quant 4bit; echo exit code $?
import json; sm = json.load(open('vlm_cache/smoke/smoke_result.json'))
print('SMOKE', sm['cells_correct'], '/', sm['cells_gt'], '| value-correct', sm['value_correct'], '|', 'passed' if sm['passed'] else 'FAILED: STOP HERE, go to step 5', '| model_revision', sm['run_info'].get('model_revision'))

## 4. Held-out v3 (only if the smoke test passed)
The script repeats the smoke test from cache and refuses to continue if it failed.

In [ ]:
assert json.load(open('vlm_cache/smoke/smoke_result.json'))['passed'], 'smoke test failed: do not run v3'
!python scripts/vlm_run.py --pdfs stress/out_heldout_v3/pdfs --cache vlm_cache --quant 4bit

## 5. Hashes and download
Prints the sha256 of every file in `vlm_cache/` and of `vlm_cache.zip`; copy this output into the chat or keep the notebook output.

In [ ]:
!find vlm_cache -type f | sort | xargs sha256sum
!zip -qr vlm_cache.zip vlm_cache && sha256sum vlm_cache.zip && ls -l vlm_cache.zip
from google.colab import files
files.download('vlm_cache.zip')